In [1]:
include("../src/main.jl");
println("MBh = $MBH")
println("MUnit = $M_unit")

Using model: iharm, change src/set_globals.jl to modify.
MBh = 6.2e9
MUnit = 3.0e26


In [2]:
#dump_filepath = "../src/models/iharm3dDumps/dump_001.h5";
dump_filepath = "../../sample_dump_SANE_a+0.94_MKS_0900.h5"
#dump_filepath = "../../../../mnt/c/Users/pedro/Downloads/torus.out0.05400.h5";

"../../sample_dump_SANE_a+0.94_MKS_0900.h5"

In [3]:
const params = read_header(dump_filepath);

Initializing grid from: ../../sample_dump_SANE_a+0.94_MKS_0900.h5


Using mixed tp_over_te with trat_small = 1, trat_large = 40, and beta_crit = 1
Using Modified Kerr-Schild coordinates MKS
MKS parameters a: 0.937500 hslope: 0.300000 Rin: 1.177706 Rout: 1000.000000
Grid start (startx): 1.635684465252566e-01, 0.000000000000000e+00, 0.000000000000000e+00 stop (stopx): 6.907755278982137e+00, 1.000000000000000e+00, 6.283185307179586e+00
grid dx: 2.341731539047528e-02, 7.812500000000000e-03, 4.908738521234052e-02


In [4]:
trat_large = 20. 
const trat_small = 1. 
const beta_crit = 1.0 
const th_beg = 1.74e-2 
const sigma_cut = 1.0 
const sigma_cut_high = -1.0;

In [5]:
const simulation_data = load_data(dump_filepath, trat_large);

Loading data from '../../sample_dump_SANE_a+0.94_MKS_0900.h5' into 'iharm' module...
All primitives successfully loaded. Dimensions: (288, 128, 128)


In [10]:
#Analytic parameters

#Setting up the parameters
#Observer distance in Rg
const ro = 1000.0
#Observer inclination in degrees
const th = 16.39958426802976

#Observer azimuth in degrees
const phi = 0.0

# Number of pixels in the x and y direction. The number of geodesics calculated will be res^2
const res = 40
const pixels_x = 40
const pixels_y = 40
# Distance to the source in parsecs
const SourceD = 16.9e6 * PC
const Rstop = 100.0
const Rh = 1 + sqrt(1. - params.a * params.a);

#Check if these are correct
const freq = 230e9;

# Size of the screen in Rg in both directions
const DXsize = SourceD/L_unit/MUAS_PER_RAD * 160
const DYsize = SourceD/L_unit/MUAS_PER_RAD * 160
# Observer fov in radians (this can be understood as size of the plane camera sees over the distance ro)
# This should be atan, but for small angles it is approximately equal to the angle itself
const fovx = DXsize/ro
const fovy = DYsize/ro
const xoff = 0.0
const yoff = 0.0


0.0

In [11]:
using ProgressMeter
include("../src/main.jl");

# Find camera in native coordinates
Xcamera = MVec4(camera_position(ro, th, phi, params.a, params.Rout))

# Scales the intensity of each pixel by the real size of each pixel
scale_factor = CalculateScaleFactor(DXsize, DYsize, pixels_x, pixels_y, SourceD, L_unit)
println("scale_factor = $scale_factor")
const maxnstep = 15000
# Generate geodesics
println("Utilizing $(Threads.nthreads()) threads for geodesic calculation.")

p = Progress(
    pixels_x * pixels_y; 
    desc = "Raytracing Image...", 
    showspeed = true, 
    barlen = 30
)
ProgressMeter.ijulia_behavior(:clear)

freq_unitless = freq * HPL/(ME * CL * CL) 
Image = zeros(Float64, pixels_x, pixels_y)
@time begin
   Threads.@threads for i in 0:(pixels_x - 1)
        tid = Threads.threadid()
        for j in 0:(pixels_y - 1)
            traj = Vector{OfTraj}()
            sizehint!(traj, maxnstep)
            nstep = get_pixel(traj, i, j, Xcamera, maxnstep, fovx, fovy, freq_unitless, pixels_x, pixels_y, params.a, Rh, params.Rout, Rstop, xoff, yoff) 
            
            resize!(traj,nstep)
            integrate_emission!(traj, nstep, Image, i + 1, j + 1, freq, params.a, simulation_data)
            ProgressMeter.next!(
                p; 
                showvalues = [
                    (:thread_id, tid), 
                    (:pixel, "($i, $j)"), 
                    (:total_done, "$(i*pixels_y + j)/$(pixels_x * pixels_y)")
                ]
            )
        end
    end
    Image *= freq^3;
end
finish!(p);

Raytracing Image... 100%|██████████████████████████████| Time: 0:00:04 ( 2.74 ms/it)
    thread_id: 1
        pixel: (39, 39)
   total_done: 1599/1600


  4.250902 seconds (45.82 M allocations: 2.491 GiB, 8.42% gc time, 33.51% compilation time)


In [8]:
OutputStokesParameters(Image, freq, scale_factor, res, SourceD)

Image processing complete. Calculating total flux and averages...
Scale = 5.876096595457265e-01
imax = 129, jmax = 154, Imax = 0.0009762617356921146, Iavg = 6.899055487791847e-6
Total Flux Fnu = 4.151246485881235e-01 Jy
nuLnu = 3.2628028913268573e40


In [38]:
OutputStokesParameters(Image, freq, scale_factor, res, SourceD)

Image processing complete. Calculating total flux and averages...
Scale = 9.401754552731624e+00
imax = 26, jmax = 35, Imax = 0.0024925484996290554, Iavg = 0.0006784620306910769
Total Flux Fnu = 4.082389430979431e+01 Jy
nuLnu = 3.208682520834291e42


In [12]:
using DelimitedFiles

writedlm("./Image_40_16.txt", Image)

In [13]:
using WGLMakie

fig = Figure(size = (600, 600))
ax = Axis(fig[1, 1], title = "Ray-traced Image", xlabel = "x (pixels)", ylabel = "y (pixels)")

# Log scale to bring out faint features
img_plot = log10.(Image))

heatmap!(ax, img_plot, colormap = :inferno)
Colorbar(fig[1, 2], colormap = :inferno, limits = extrema(img_plot), label = "log₁₀ Intensity")

display(fig)

WGLMakie.Screen(
    framerate = 30.0,
    resize_to = nothing,
    px_per_unit = automatic,
    scalefactor = automatic,
    session = false,
    three = Channel{Any}(1) (empty),
    scene = Scene(2 children, 0 plots),
)